# 05 — PEP 703 : Free-Threaded Python (3.13+/3.14)

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- expliquer le contenu de la PEP 703 et le projet no-GIL ;
- vérifier si votre interpréteur est compilé en mode free-threaded ;
- comprendre le nouveau modèle de comptage de références (biased ref counting) ;
- mesurer le vrai parallélisme CPU avec des threads en mode free-threaded ;
- connaître les limitations actuelles et la roadmap.

## Prérequis — ce que vous connaissez déjà

Ce notebook s'adresse à un développeur Python **confirmé**. Vous maîtrisez déjà :

- `threading.Thread`, `Lock`, les primitives de synchronisation ;
- le GIL, son impact sur les tâches CPU-bound ;
- `multiprocessing` comme solution de contournement ;
- `concurrent.futures` pour l'abstraction de haut niveau.

Notions que nous allons **introduire** ici :

- PEP 703 (Making the Global Interpreter Lock Optional) ;
- le mode **free-threaded** de CPython 3.13t/3.14t ;
- les changements internes : biased ref counting, per-object locks ;
- l'impact sur le code existant et les extensions C.

## Plan

1. Rappel : pourquoi le GIL pose problème
2. PEP 703 — historique et décision
3. Détecter le mode free-threaded
4. Ce qui change en interne
5. Benchmark : threads CPU-bound avec et sans GIL
6. Impact sur le code existant
7. Impact sur les extensions C
8. Roadmap et état actuel
9. Synthèse
10. Exercices

---

## 1. Rappel : pourquoi le GIL pose problème

Le **GIL** (Global Interpreter Lock) est un verrou qui empêche l'exécution simultanée de bytecode Python par plusieurs threads. Conséquences :

- Les tâches **CPU-bound** ne bénéficient pas du multi-threading.
- Il faut passer par `multiprocessing` (coûteux en mémoire, sérialisation).
- Python est désavantagé face à Java, Go, Rust pour le calcul parallèle.

Le GIL existe depuis 1992 et de nombreuses tentatives de suppression ont échoué (projet Gilectomy de Larry Hastings, 2016). La PEP 703 est la première à être **acceptée**.

---

## 2. PEP 703 — historique et décision

| Date | Événement |
|---|---|
| 2023-01 | Sam Gross soumet la PEP 703 |
| 2023-07 | Le Steering Council accepte la PEP |
| 2023-10 | Python 3.13 alpha inclut le build `--disable-gil` |
| 2024-10 | Python 3.13.0 sort avec le mode free-threaded **expérimental** |
| 2025-04 | Python 3.14 stabilise le mode (toujours opt-in) |
| ~2027 | Objectif : free-threaded par défaut (si la communauté est prête) |

**Le plan en 3 phases :**

1. **Phase 1** (3.13) : build expérimental séparé (`python3.13t`)
2. **Phase 2** (3.14-3.16) : build supporté, GIL désactivable à l'exécution
3. **Phase 3** (~3.17+) : free-threaded par défaut, GIL supprimé

---

## 3. Détecter le mode free-threaded

Depuis Python 3.13, on peut vérifier avec `sys._is_gil_enabled()`.

In [ ]:
import sys

print(f"Version Python : {sys.version}")
print(f"Build info : {sys.version_info}")

# Disponible depuis Python 3.13
if hasattr(sys, '_is_gil_enabled'):
    print(f"GIL activé : {sys._is_gil_enabled()}")
else:
    print("sys._is_gil_enabled() non disponible (Python < 3.13)")

In [ ]:
import sysconfig

# Vérifier si le build est free-threaded
gil_disabled = sysconfig.get_config_var('Py_GIL_DISABLED')
print(f"Py_GIL_DISABLED = {gil_disabled}")
print(f"Build free-threaded : {'oui' if gil_disabled else 'non'}")

### Installer un Python free-threaded

```bash
# Avec pyenv
pyenv install 3.14.0t

# Avec uv
uv python install 3.14t

# Vérification
python3.14t -c "import sys; print(sys._is_gil_enabled())"
# False
```

---

## 4. Ce qui change en interne

Pour retirer le GIL sans casser CPython, Sam Gross a dû résoudre plusieurs problèmes fondamentaux :

### 4.1. Biased Reference Counting

Le comptage de références classique (`ob_refcnt`) est **non thread-safe** : deux threads qui incrémentent/décrémentent en même temps perdent des mises à jour. Le GIL protégeait cela.

Solution : un compteur **biaisé** vers le thread créateur. Ce thread manipule le compteur sans opération atomique (rapide). Les autres threads utilisent un compteur séparé avec des opérations atomiques.

In [ ]:
import sys

# sys.getrefcount() fonctionne toujours
x = [1, 2, 3]
print(f"Refcount de x : {sys.getrefcount(x)}")
# Note : getrefcount ajoute 1 (la référence temporaire dans l'appel)

### 4.2. Per-Object Locks

Au lieu d'un verrou global, chaque objet possède son propre **verrou léger**. Quand une opération doit modifier un objet (comme `dict.__setitem__`), elle prend le verrou de cet objet seulement.

### 4.3. Collections thread-safe

Les opérations atomiques sur les types built-in (`list.append`, `dict[key] = val`, `set.add`) restent thread-safe en mode free-threaded, grâce aux per-object locks.

In [ ]:
import threading

# list.append est toujours atomique
resultats: list[int] = []

def ajouter(n: int) -> None:
    for i in range(n):
        resultats.append(i)  # atomique, même sans GIL

threads = [threading.Thread(target=ajouter, args=(10_000,)) for _ in range(4)]
for t in threads: t.start()
for t in threads: t.join()
print(f"Éléments : {len(resultats)} (attendu : 40_000)")

### 4.4. Immortalization

Certains objets (singletons comme `None`, `True`, `False`, les petits entiers) sont rendus **immortels** : leur refcount n'est jamais modifié. Cela élimine la contention sur ces objets très fréquemment accédés.

In [ ]:
import sys

# Les objets immortels ont un refcount spécial
print(f"refcount(None)  : {sys.getrefcount(None)}")
print(f"refcount(True)  : {sys.getrefcount(True)}")
print(f"refcount(0)     : {sys.getrefcount(0)}")
# Ces valeurs sont très grandes car les objets sont immortels

---

## 5. Benchmark : threads CPU-bound avec et sans GIL

Le test ultime : est-ce que les threads Python peuvent maintenant accélérer du calcul CPU ?

In [ ]:
import threading
import time
import sys

def calcul_cpu(n: int) -> int:
    """Tâche CPU-bound pure."""
    total = 0
    for i in range(n):
        total += i * i
    return total

N = 5_000_000
NB_THREADS = 4

# 1. Séquentiel
start = time.perf_counter()
for _ in range(NB_THREADS):
    calcul_cpu(N)
t_seq = time.perf_counter() - start

# 2. Multithreadé
start = time.perf_counter()
threads = [threading.Thread(target=calcul_cpu, args=(N,)) for _ in range(NB_THREADS)]
for t in threads: t.start()
for t in threads: t.join()
t_thr = time.perf_counter() - start

gil_status = "désactivé" if hasattr(sys, '_is_gil_enabled') and not sys._is_gil_enabled() else "activé"
print(f"GIL : {gil_status}")
print(f"Séquentiel  : {t_seq:.3f}s")
print(f"Threading   : {t_thr:.3f}s")
print(f"Speedup     : {t_seq / t_thr:.2f}x")

if t_seq / t_thr > 1.5:
    print("Vrai parallélisme CPU avec les threads !")
else:
    print("Pas de parallélisme CPU (GIL actif).")

### Résultats attendus

| Mode | Speedup (4 threads, 4 cores) |
|---|---|
| Python classique (GIL) | ~1.0x (pas d'accélération) |
| Python free-threaded | ~3.5-4.0x (vrai parallélisme) |

---

## 6. Impact sur le code existant

### 6.1. Le code correct reste correct

Si votre code Python utilise correctement les verrous (`Lock`, `Queue`, `Condition`...), il fonctionne **identiquement** avec ou sans GIL.

### 6.2. Le code qui « marchait par accident » peut casser

Certains programmes fonctionnaient sans verrous **uniquement grâce au GIL**. En mode free-threaded, les race conditions deviennent réelles.

In [ ]:
import threading

# Ce code a TOUJOURS été bugué, mais le GIL masquait souvent le problème
compteur = 0

def incrementer_buggy(n: int) -> None:
    global compteur
    for _ in range(n):
        compteur += 1  # read-modify-write non atomique

threads = [threading.Thread(target=incrementer_buggy, args=(100_000,)) for _ in range(4)]
for t in threads: t.start()
for t in threads: t.join()

print(f"Résultat : {compteur} (attendu : 400_000)")
print("En mode free-threaded, l'écart sera potentiellement plus grand.")

### 6.3. `PYTHON_GIL=1` pour forcer le GIL

Si du code legacy pose problème, on peut réactiver le GIL :

```bash
PYTHON_GIL=1 python3.14t mon_script.py
```

Ou en Python :

```python
import sys
if hasattr(sys, '_is_gil_enabled'):
    print(f"GIL activé : {sys._is_gil_enabled()}")
```

---

## 7. Impact sur les extensions C

C'est le **plus gros défi** du projet free-threaded. Les extensions C (NumPy, pandas, etc.) utilisaient le GIL comme protection implicite.

### Marquage des extensions

Chaque extension C doit déclarer si elle est compatible free-threaded :

```c
// Dans le module init
Py_mod_gil = Py_MOD_GIL_NOT_USED  // compatible
```

Si une extension n'est pas marquée, Python **réactive automatiquement le GIL** pour protéger le code legacy.

### État des grandes bibliothèques (avril 2025)

| Bibliothèque | Statut free-threaded |
|---|---|
| NumPy | Compatible depuis 2.1 |
| Cython | Support en cours |
| pydantic | Compatible |
| SQLAlchemy | En cours d'adaptation |
| PyTorch | Support expérimental |

---

## 8. Roadmap et état actuel

### Python 3.13 (octobre 2024)
- Build expérimental `python3.13t`
- `--disable-gil` flag
- Performance : ~5-10% plus lent en single-thread

### Python 3.14 (octobre 2025)
- Build plus stable
- Performance single-thread améliorée (~2-5% overhead)
- Plus de bibliothèques compatibles
- `sys._is_gil_enabled()` API stable

### Python 3.15-3.16 (futur)
- Réduction de l'overhead single-thread
- Objectif : overhead < 2%
- Plus de bibliothèques C portées

### Python ~3.17+ (objectif)
- Free-threaded par défaut
- GIL supprimé du build standard

---

## 9. Synthèse

| Aspect | Détail |
|---|---|
| PEP 703 | Rend le GIL optionnel |
| Build | `python3.14t` (suffixe `t`) |
| Détection | `sys._is_gil_enabled()` |
| Mécanisme | Biased ref counting + per-object locks |
| Bénéfice | Vrai parallélisme CPU avec des threads |
| Overhead | ~2-5% en single-thread (3.14) |
| Extensions C | Doivent être marquées compatible |
| Réactivation | `PYTHON_GIL=1` ou automatique si extension incompatible |

**Recommandations immédiates :**

1. Écrivez du code thread-safe **dès maintenant** (verrous, queues).
2. Ne comptez pas sur le GIL pour la synchronisation.
3. Testez votre code avec `python3.14t` pour anticiper.
4. Si vous écrivez des extensions C, ajoutez `Py_MOD_GIL_NOT_USED`.

---

## 10. Exercices

### Exercice 1 — Détecter le mode GIL *(facile)*

Écrire une fonction `diagnostique_gil()` qui affiche :

- La version Python.
- Si `sys._is_gil_enabled()` est disponible et sa valeur.
- La valeur de `sysconfig.get_config_var('Py_GIL_DISABLED')`.
- Une recommandation ("Vous pouvez utiliser le threading pour le CPU" ou "Utilisez multiprocessing pour le CPU").

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="05_PEP703_free_threaded", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
import sys
import sysconfig

def diagnostique_gil() -> None:
    print(f"Python {sys.version}")

    if hasattr(sys, '_is_gil_enabled'):
        gil = sys._is_gil_enabled()
        print(f"GIL activé : {gil}")
    else:
        gil = True
        print("sys._is_gil_enabled() non disponible → GIL présumé actif")

    disabled = sysconfig.get_config_var('Py_GIL_DISABLED')
    print(f"Py_GIL_DISABLED : {disabled}")

    if not gil:
        print("→ Mode free-threaded : threading exploite le CPU !")
    else:
        print("→ GIL actif : utilisez multiprocessing pour le CPU.")

diagnostique_gil()
```

</details>

### Exercice 2 — Benchmark comparatif *(moyen)*

Écrire un benchmark qui compare, pour une tâche CPU (somme des carrés de 0 à N) :

1. Exécution séquentielle.
2. 4 threads.
3. 4 processus (via `multiprocessing`).

Afficher les temps et les speedups. Conclure si le Python utilisé est free-threaded ou non.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="05_PEP703_free_threaded", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
import threading
import multiprocessing
import time
import sys

def somme_carres(n: int = 5_000_000) -> int:
    return sum(i * i for i in range(n))

NB = 4

# Séquentiel
t0 = time.perf_counter()
for _ in range(NB):
    somme_carres()
t_seq = time.perf_counter() - t0

# Threading
t0 = time.perf_counter()
threads = [threading.Thread(target=somme_carres) for _ in range(NB)]
for t in threads: t.start()
for t in threads: t.join()
t_thr = time.perf_counter() - t0

# Multiprocessing
t0 = time.perf_counter()
procs = [multiprocessing.Process(target=somme_carres) for _ in range(NB)]
for p in procs: p.start()
for p in procs: p.join()
t_mp = time.perf_counter() - t0

print(f"Séquentiel     : {t_seq:.3f}s (ref)")
print(f"Threading      : {t_thr:.3f}s ({t_seq/t_thr:.2f}x)")
print(f"Multiprocessing: {t_mp:.3f}s ({t_seq/t_mp:.2f}x)")

if t_seq / t_thr > 1.5:
    print("\nConclusion : free-threaded actif !")
else:
    print("\nConclusion : GIL actif, threading n'accélère pas le CPU.")
```

</details>

### Exercice 3 — Trouver les race conditions *(difficile)*

Le code ci-dessous contient **3 race conditions** qui passent souvent inaperçues avec le GIL mais deviennent problématiques en mode free-threaded. Identifiez-les et corrigez-les.

```python
import threading

class Banque:
    def __init__(self):
        self.comptes = {"A": 1000, "B": 1000}

    def transferer(self, src, dst, montant):
        if self.comptes[src] >= montant:           # race 1 : check-then-act
            self.comptes[src] -= montant            # race 2 : non-atomic RMW
            self.comptes[dst] += montant            # race 3 : non-atomic RMW

    def total(self):
        return sum(self.comptes.values())
```

In [ ]:
# Votre code corrigé ici


<details>
<summary>📖 Voir la correction</summary>

```python
import threading

class BanqueSafe:
    def __init__(self) -> None:
        self.comptes = {"A": 1000, "B": 1000}
        self._lock = threading.Lock()

    def transferer(self, src: str, dst: str, montant: int) -> bool:
        with self._lock:  # protège check + modification
            if self.comptes[src] >= montant:
                self.comptes[src] -= montant
                self.comptes[dst] += montant
                return True
            return False

    def total(self) -> int:
        with self._lock:
            return sum(self.comptes.values())

banque = BanqueSafe()

def stress(n: int) -> None:
    for _ in range(n):
        banque.transferer("A", "B", 1)
        banque.transferer("B", "A", 1)

threads = [threading.Thread(target=stress, args=(10_000,)) for _ in range(4)]
for t in threads: t.start()
for t in threads: t.join()

print(f"Total : {banque.total()} (doit être 2000)")
print(f"A={banque.comptes['A']}, B={banque.comptes['B']}")
```

Les 3 race conditions :

1. **Check-then-act** : entre la vérification (`>= montant`) et la modification, un autre thread peut retirer de l'argent.
2. **Read-modify-write sur src** : `comptes[src] -= montant` = read + sub + write.
3. **Read-modify-write sur dst** : `comptes[dst] += montant` = read + add + write.

Solution : un seul `Lock` autour de l'ensemble check + modifications.

</details>

---

## Ressources

- [PEP 703 — Making the Global Interpreter Lock Optional](https://peps.python.org/pep-0703/)
- [Sam Gross — nogil project](https://github.com/colesbury/nogil)
- [Python 3.13 Release Notes — Free-threaded CPython](https://docs.python.org/3.13/whatsnew/3.13.html#free-threaded-cpython)
- [Python 3.14 Release Notes](https://docs.python.org/3.14/whatsnew/3.14.html)
- [py-free-threading.github.io](https://py-free-threading.github.io/) — tracker communautaire